<div align="center">
<h1 style="color:rgb(121, 65, 6);">Speech Emotion Recognition</h1>
<h3>Prediction — Classify a new audio file</h3>
</div>

**Description:** Load the saved model and scaler, extract features from a new audio file, and predict the emotion.

## Setup

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd

from src.config import MODELS_DIR, SR, OFFSET, N_MFCC
from src.features import extract_features, get_feature_names

# Load the saved model and scaler
model  = joblib.load(MODELS_DIR / "best_model.joblib")
scaler = joblib.load(MODELS_DIR / "scaler.joblib")

# The class labels the model was trained on (sorted, as sklearn does internally)
class_labels = list(model.classes_)
print(f"Model type : {type(model).__name__}")
print(f"Classes    : {class_labels}")
print(f"Features   : {len(get_feature_names())}")
print("\nModel and scaler loaded successfully!")

## Choose an audio file

Set `AUDIO_PATH` to the audio file you want to classify. It can be any `.wav` file.

In [ ]:
# ── Set your audio file path here ──────────────────────────────────────
AUDIO_PATH = "../data/raw/cremad/versions/1/AudioWAV/1003_TAI_ANG_XX.wav"
# ──────────────────────────────────────────────────────────────────────────

## Visualize the audio

In [ ]:
y_audio, sr_audio = librosa.load(AUDIO_PATH, sr=SR)

print(f"File     : {AUDIO_PATH}")
print(f"Duration : {librosa.get_duration(y=y_audio, sr=sr_audio):.2f}s")
print(f"SR       : {sr_audio} Hz")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Waveform
librosa.display.waveshow(y_audio, sr=sr_audio, ax=axes[0])
axes[0].set_title("Waveform")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

# Mel spectrogram
S = librosa.feature.melspectrogram(y=y_audio, sr=sr_audio)
S_dB = librosa.power_to_db(S, ref=np.max)
img = librosa.display.specshow(S_dB, sr=sr_audio, x_axis='time', y_axis='mel', ax=axes[1])
axes[1].set_title("Mel Spectrogram")
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')

plt.tight_layout()
plt.show()

ipd.Audio(AUDIO_PATH)

## Extract features & predict

In [ ]:
# Extract the same features used during training
features = extract_features(AUDIO_PATH)
feature_names = get_feature_names()

# Wrap in a DataFrame with proper column names to avoid sklearn warnings
features_df = pd.DataFrame(features.reshape(1, -1), columns=feature_names)

# Scale with the same scaler used during training
features_scaled = pd.DataFrame(
    scaler.transform(features_df),
    columns=feature_names,
)

# Predict
prediction = model.predict(features_scaled)[0]

# If the model supports probabilities, show them
if hasattr(model, "predict_proba"):
    probas = model.predict_proba(features_scaled)[0]
    proba_df = pd.DataFrame({
        "class": class_labels,
        "probability": probas
    }).sort_values("probability", ascending=False).reset_index(drop=True)

gender, emotion = prediction.split("_")

print("═" * 45)
print(f"  Predicted class : {prediction}")
print(f"  Gender          : {gender}")
print(f"  Emotion         : {emotion}")
if hasattr(model, "predict_proba"):
    confidence = probas[class_labels.index(prediction)] * 100
    print(f"  Confidence      : {confidence:.1f}%")
print("═" * 45)

## Probability distribution

In [ ]:
if hasattr(model, "predict_proba"):
    colors = ["#2ecc71" if c == prediction else "#95a5a6" for c in proba_df["class"]]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(proba_df["class"], proba_df["probability"], color=colors,
                   edgecolor="black", linewidth=0.4)
    ax.set_xlabel("Probability", fontsize=12)
    ax.set_title(f"Prediction probabilities — predicted: {prediction}", fontsize=13)
    ax.set_xlim(0, 1)
    ax.invert_yaxis()

    for bar, prob in zip(bars, proba_df["probability"]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{prob:.1%}", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("Model does not support predict_proba — no probability chart available.")

## Try your own audio!

Go back to the **"Choose an audio file"** cell, change `AUDIO_PATH` to any `.wav` file, and re-run the cells below it.